# SaemPhysics: Render a Lesson Preview in Google Colab

Run the cells from top to bottom. This notebook checks out the repository, installs Manim's system and Python dependencies, **uploads your private voice sample**, validates the batch manifest, and renders a short MP4 preview. 

All outputs are saved in the Colab runtime's local `/content/physics` folder. Download anything you want to keep before the runtime is deleted. Generated visuals are drafts; physics and release approval remain human decisions.

In [ ]:
from pathlib import Path
import os
import subprocess

# UPDATE THIS if your repo URL or branch changes
REPOSITORY = "https://github.com/saemsir/physics.git"
REVISION = "main"
ROOT = Path("/content/physics")

if (ROOT / ".git").is_dir():
    print(f"Updating existing checkout at {ROOT}")
    subprocess.run(["git", "-C", str(ROOT), "pull"], check=True)
elif ROOT.exists() and any(ROOT.iterdir()):
    raise RuntimeError(f"{ROOT} exists but is not a Git checkout; move or rename it, then rerun.")
else:
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(ROOT)], check=True)

subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", REVISION], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
print(f"✅ Repository ready at {ROOT}")

## 🔒 Upload Private Voice Sample

Because `saem_voice_sample.wav` is in `.gitignore` to protect your privacy, you must upload it manually to this Colab session for offline voice cloning to work.

In [ ]:
from google.colab import files
import shutil

VOICE_SAMPLE_PATH = ROOT / "saem_voice_sample.wav"
if not VOICE_SAMPLE_PATH.exists():
    print("🎤 Please upload your voice sample (saem_voice_sample.wav) when prompted.")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.lower() == "saem_voice_sample.wav":
            shutil.move(filename, VOICE_SAMPLE_PATH)
            print(f"✅ Moved {filename} to {VOICE_SAMPLE_PATH}")
        else:
            print(f"⚠️ Ignored {filename}. Please upload 'saem_voice_sample.wav'")
else:
    print("✅ Voice sample already exists in the repository.")

## Install Render Dependencies

A fresh Colab runtime needs FFmpeg, Cairo, Pango, and LaTeX for Manim. This setup takes 2-4 minutes.

In [ ]:
import sys

print("Installing system dependencies...")
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run([
    "apt-get", "-qq", "install", "-y",
    "ffmpeg", "libcairo2-dev", "libpango1.0-dev",
    "texlive-latex-extra", "texlive-fonts-recommended", "dvisvgm",
], check=True)

print("Installing Python dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt"),
], check=True)

print("✅ Verifying installations...")
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "manim", "--version"], check=True)

## Validate and Compile the Lesson

*Note: Ensure `content_engine` matches your actual CLI module name (e.g., `matrix_pipeline`, `exam_engine`, or `main`).* 

In [ ]:
import json

# UPDATE THESE PATHS to match your actual manifest and module names
CLI_MODULE = "content_engine"  # Change to 'matrix_pipeline' or 'main' if refactored
BATCH_MANIFEST = ROOT / "data/video_batches/batch_002.json"
BATCH_PLAN = Path("/content/batch_plan.json")
DATABASE = Path("/content/saemphysics.sqlite")
LESSON_IR = Path("/content/kinematics-lesson.json")

print("Validating batch plan...")
subprocess.run([
    sys.executable, "-m", CLI_MODULE, "batch-plan",
    "--manifest", str(BATCH_MANIFEST), "--output", str(BATCH_PLAN),
], cwd=ROOT, check=True)

batch_plan = json.loads(BATCH_PLAN.read_text(encoding="utf-8"))
batch_manifest = json.loads(BATCH_MANIFEST.read_text(encoding="utf-8"))
declared_formats = [item["format"] for item in batch_manifest.get("video_outputs", [])]
expected_deliverables = len(batch_manifest.get("episodes", [])) * len(declared_formats)

if len(batch_plan.get("deliverables", [])) != expected_deliverables:
    print(f"⚠️ Warning: Expected {expected_deliverables} deliverables, found {len(batch_plan.get('deliverables', []))}")

print("Compiling lesson...")
subprocess.run([
    sys.executable, "-m", CLI_MODULE, "init", "--database", str(DATABASE),
], cwd=ROOT, check=True)

subprocess.run([
    sys.executable, "-m", CLI_MODULE, "compile-lesson",
    "--database", str(DATABASE), "--problem", str(ROOT / "data/kinematics_demo.json"),
    "--output", str(LESSON_IR),
], cwd=ROOT, check=True)

lesson = json.loads(LESSON_IR.read_text(encoding="utf-8"))
print(f"✅ Compiled: {lesson.get('title', 'Unknown')}; batch deliverables validated: {len(batch_plan.get('deliverables', []))}")

## Render One Preview Locally

This renders a low-resolution (`-ql`) preview to verify the animation and audio stitching.

In [ ]:
OUTPUT_ROOT = Path("/content/physics/previews")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MEDIA_ROOT = Path("/content/physics-render")

render_env = os.environ.copy()
render_env["SAEMPHYSICS_LESSON_IR"] = str(LESSON_IR)

# UPDATE 'episodes/kinematics_001/scenes.py' and 'KinematicsMotion' to match your actual scene file and class name
render_command = [
    sys.executable, "-m", "manim", "-ql",
    "--media_dir", str(MEDIA_ROOT), "--format", "mp4", "--disable_caching",
    str(ROOT / "episodes/kinematics_001/scenes.py"), "KinematicsMotion",
]
print("🎬 Rendering preview...")
subprocess.run(render_command, cwd=ROOT, env=render_env, check=True)

renders = list(MEDIA_ROOT.rglob("KinematicsMotion.mp4"))
if len(renders) != 1 or renders[0].stat().st_size == 0:
    raise FileNotFoundError(f"Expected one non-empty KinematicsMotion.mp4, found {len(renders)}")

rendered = renders[0]
probe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries",
    "format=duration:stream=width,height", "-of", "json", str(rendered),
], check=True, capture_output=True, text=True)

video_info = json.loads(probe.stdout)
video_stream = next((stream for stream in video_info["streams"] if "width" in stream), None)

import shutil
OUTPUT_VIDEO = OUTPUT_ROOT / "kinematics-demo-preview.mp4"
shutil.copy2(rendered, OUTPUT_VIDEO)
print(f"✅ Verified {video_stream['width']}x{video_stream['height']} video, {float(video_info['format']['duration']):.2f}s")
print(f"Preview saved at: {OUTPUT_VIDEO}")

In [ ]:
from IPython.display import Video, display
display(Video(str(OUTPUT_VIDEO), embed=True, width=640))

## Optional: Render the Full Batch

Set `RENDER_FULL_BATCH = True` and `FORCE_RENDER = True` (if your manifest status is still 'draft') to render all videos. This may take a long time.

In [ ]:
RENDER_FULL_BATCH = False
FORCE_RENDER = True  # Set to True to bypass 'approved-for-storyboarding' check

if RENDER_FULL_BATCH:
    if batch_plan.get("status") != "approved-for-storyboarding" and not FORCE_RENDER:
        raise ValueError("High-quality rendering requires storyboard approval (or set FORCE_RENDER = True).")
    
    print("🚀 Rendering full batch...")
    force_flag = ["--force"] if FORCE_RENDER else []
    
    subprocess.run([
        sys.executable, str(ROOT / "tools/render_video_batch.py"),
        "--manifest", str(BATCH_MANIFEST), "--quality", "high",
        "--formats", *declared_formats, "--resume", *force_flag,
        "--output-dir", str(OUTPUT_ROOT.parent / f"{batch_plan.get('batch_id', 'batch')}_high"),
    ], cwd=ROOT, check=True)
    print("✅ Full batch rendering complete!")
else:
    print("⏭️ Full batch skipped. Set RENDER_FULL_BATCH = True after reviewing the preview.")

## Download Outputs

Run this cell to display clickable download links. Save the files to your computer before closing the tab, as Colab will delete them when the runtime disconnects.

In [ ]:
from IPython.display import FileLink, display
import glob

if not OUTPUT_VIDEO.is_file():
    raise FileNotFoundError(f"Preview not found: {OUTPUT_VIDEO}. Run the preview cells first.")

download_paths = [OUTPUT_VIDEO]
batch_output = OUTPUT_ROOT.parent / f"{batch_plan.get('batch_id', 'batch')}_high"

if batch_output.is_dir():
    download_paths.extend(sorted(batch_output.glob("*.mp4")))
    report = batch_output / "render_report.json"
    if report.is_file():
        download_paths.append(report)

print(f"📦 {len(download_paths)} file(s) available to download:")
for path in download_paths:
    display(FileLink(str(path)))